## Main focus is to `ingest data from Kafka cluster(Broker)` to the databricks and `save the raw data` to the `Bronze layar delta table`

### For the library to connect kafka with databricks/pyspark use this page
  - link: use `Maven artifact`: https://spark.apache.org/docs/latest/structured-streaming-kafka-integration.html

- Here, only the code for `consuming the data` from Kafka cluster(broker) is given
- for the producer code reflect to the repo link given below:
    - `link` to repo containing `kafka producer` source code: 


In [0]:
import time 
import json

In [0]:
bootstrap_server_url="pkc-p11xm.us-east-1.aws.confluent.cloud:9092"
jaas_module="org.apache.kafka.common.security.plain.PlainLoginModule"
cluster_api_key="WEB7EAYJ2KYZBDSJ"
cluster_api_secrete="MTkdyoPgsQhKZ/MhXIAnRavB75jS0mZzXwDuT2+zaYYvHIof6pdtFT0lm5efqbTP"
topic="invoice"
project_dir="/dbfs/FileStore/project5/"


In [0]:
from pyspark.sql.functions import decode, col, expr

In [0]:
class KafkaDataIngestionBronzeLayer:
    def __init__(self, project_dir:str, topic:str, bootstrap_server_url:str, cluster_api_key:str, cluster_api_secrete:str)->None:
        self.bronzeLayerTableName="project5_kafka_bronze_layer_data"
        self.project_dir=project_dir
        self.topic=topic
        self.jaas_module="org.apache.kafka.common.security.plain.PlainLoginModule"
        self.bootstrap_server_url=bootstrap_server_url
        self.cluster_api_key=cluster_api_key
        self.cluster_api_secrete=cluster_api_secrete

    def clenup_n_setup(self):
        drop_query=f"drop table if exists {self.bronzeLayerTableName}"
        spark.sql(drop_query)
        dbutils.fs.rm(f"/user/hive/warehouse/{self.bronzeLayerTableName}",True)


    def ingest_data_from_kafka(self):
        df=spark.read.format("kafka")\
            .option("kafka.bootstrap.servers",self.bootstrap_server_url)\
                .option("kafka.security.protocol","SASL_SSL")\
                    .option("kafka.sasl.mechanism", "PLAIN")\
                        .option("kafka.sasl.jaas.config",f"{self.jaas_module} required username='{self.cluster_api_key}' password='{self.cluster_api_secrete}';")\
                            .option("subscribe",self.topic)\
                                .load()
        df=df.withColumn("value", decode(col("value"), "UTF-8"))
        df=df.select("value")
        df.write.format("delta").mode("overwrite").saveAsTable(self.bronzeLayerTableName)


    def startIngestion(self):

        self.clenup_n_setup()
        print("CLEAN_UP AND SET_UP COMPLETED !")
        
        print("Data ingestion started..")
        self.ingest_data_from_kafka()
        print(f"DATA INGETSTION FROM KAFKA TOPIC: {self.topic} COMPLETED !")
        print("SAMPLE DATA::")
        display(spark.table(self.bronzeLayerTableName).head(30))


        print("PROCESS COMPETED !")

        

In [0]:
kafkaDataIngestion=KafkaDataIngestionBronzeLayer(project_dir=project_dir,topic=topic, bootstrap_server_url=bootstrap_server_url, cluster_api_key=cluster_api_key,cluster_api_secrete=cluster_api_secrete)
kafkaDataIngestion.startIngestion()


CLEAN_UP AND SET_UP COMPLETED !
Data ingestion started..
DATA INGETSTION FROM KAFKA TOPIC: invoice COMPLETED !
SAMPLE DATA::


value
"{""InvoiceNumber"":""51402977"",""CreatedTime"":1595688900348,""StoreID"":""STR7188"",""PosID"":""POS956"",""CashierID"":""OAS134"",""CustomerType"":""PRIME"",""CustomerCardNo"":""4629185211"",""TotalAmount"":11114.0,""NumberOfItems"":4,""PaymentMethod"":""CARD"",""TaxableAmount"":11114.0,""CGST"":277.85,""SGST"":277.85,""CESS"":13.8925,""DeliveryType"":""TAKEAWAY"",""InvoiceLineItems"":[{""ItemCode"":""458"",""ItemDescription"":""Wine glass"",""ItemPrice"":1644.0,""ItemQty"":2,""TotalValue"":3288.0},{""ItemCode"":""283"",""ItemDescription"":""Portable Lamps"",""ItemPrice"":2236.0,""ItemQty"":1,""TotalValue"":2236.0},{""ItemCode"":""498"",""ItemDescription"":""Carving knifes"",""ItemPrice"":1424.0,""ItemQty"":2,""TotalValue"":2848.0},{""ItemCode"":""523"",""ItemDescription"":""Oil-lamp clock"",""ItemPrice"":1371.0,""ItemQty"":2,""TotalValue"":2742.0}]}"
"{""InvoiceNumber"":""8320594"",""CreatedTime"":1595688902254,""StoreID"":""STR7188"",""PosID"":""POS825"",""CashierID"":""OAS329"",""CustomerType"":""PRIME"",""CustomerCardNo"":""7051101351"",""TotalAmount"":5824.0,""NumberOfItems"":3,""PaymentMethod"":""CASH"",""TaxableAmount"":5824.0,""CGST"":145.6,""SGST"":145.6,""CESS"":7.28,""DeliveryType"":""HOME-DELIVERY"",""DeliveryAddress"":{""AddressLine"":""2465 Laoreet, Street"",""City"":""Dehri"",""State"":""Bihar"",""PinCode"":""637308"",""ContactNumber"":""2662305605""},""InvoiceLineItems"":[{""ItemCode"":""288"",""ItemDescription"":""Hutch"",""ItemPrice"":1812.0,""ItemQty"":2,""TotalValue"":3624.0},{""ItemCode"":""558"",""ItemDescription"":""Balloon clock"",""ItemPrice"":1633.0,""ItemQty"":1,""TotalValue"":1633.0},{""ItemCode"":""658"",""ItemDescription"":""Chinois"",""ItemPrice"":567.0,""ItemQty"":1,""TotalValue"":567.0}]}"
"{""InvoiceNumber"":""26723058"",""CreatedTime"":1595689028262,""StoreID"":""STR7188"",""PosID"":""POS664"",""CashierID"":""OAS971"",""CustomerType"":""PRIME"",""CustomerCardNo"":""9316477281"",""TotalAmount"":5235.0,""NumberOfItems"":3,""PaymentMethod"":""CARD"",""TaxableAmount"":5235.0,""CGST"":130.875,""SGST"":130.875,""CESS"":6.54375,""DeliveryType"":""HOME-DELIVERY"",""DeliveryAddress"":{""AddressLine"":""5418 Magna. Rd."",""City"":""Chennai"",""State"":""Tamil Nadu"",""PinCode"":""386032"",""ContactNumber"":""6557358508""},""InvoiceLineItems"":[{""ItemCode"":""653"",""ItemDescription"":""Browning tray"",""ItemPrice"":375.0,""ItemQty"":1,""TotalValue"":375.0},{""ItemCode"":""568"",""ItemDescription"":""Pinch Pleated Curtains"",""ItemPrice"":1718.0,""ItemQty"":2,""TotalValue"":3436.0},{""ItemCode"":""498"",""ItemDescription"":""Carving knifes"",""ItemPrice"":1424.0,""ItemQty"":1,""TotalValue"":1424.0}]}"
"{""InvoiceNumber"":""51402977"",""CreatedTime"":1595688900348,""StoreID"":""STR7188"",""PosID"":""POS956"",""CashierID"":""OAS134"",""CustomerType"":""PRIME"",""CustomerCardNo"":""4629185211"",""TotalAmount"":11114.0,""NumberOfItems"":4,""PaymentMethod"":""CARD"",""TaxableAmount"":11114.0,""CGST"":277.85,""SGST"":277.85,""CESS"":13.8925,""DeliveryType"":""TAKEAWAY"",""InvoiceLineItems"":[{""ItemCode"":""458"",""ItemDescription"":""Wine glass"",""ItemPrice"":1644.0,""ItemQty"":2,""TotalValue"":3288.0},{""ItemCode"":""283"",""ItemDescription"":""Portable Lamps"",""ItemPrice"":2236.0,""ItemQty"":1,""TotalValue"":2236.0},{""ItemCode"":""498"",""ItemDescription"":""Carving knifes"",""ItemPrice"":1424.0,""ItemQty"":2,""TotalValue"":2848.0},{""ItemCode"":""523"",""ItemDescription"":""Oil-lamp clock"",""ItemPrice"":1371.0,""ItemQty"":2,""TotalValue"":2742.0}]}"
"{""InvoiceNumber"":""8320594"",""CreatedTime"":1595688902254,""StoreID"":""STR7188"",""PosID"":""POS825"",""CashierID"":""OAS329"",""CustomerType"":""PRIME"",""CustomerCardNo"":""7051101351"",""TotalAmount"":5824.0,""NumberOfItems"":3,""PaymentMethod"":""CASH"",""TaxableAmount"":5824.0,""CGST"":145.6,""SGST"":145.6,""CESS"":7.28,""DeliveryType"":""HOME-DELIVERY"",""DeliveryAddress"":{""AddressLine"":""2465 Laoreet, Street"",""City"":""Dehri"",""

PROCESS COMPETED !
